# totalVI MALT CITE-seq analysis

This notebook loads a 10x Genomics CITE-seq dataset and prepares it for analysis with totalVI.

## 1. Import libraries

In [1]:
import os
import tempfile
import requests

import numpy as np
import pandas as pd
import scipy

import matplotlib.pyplot as plt
import seaborn as sns

import scanpy as sc
import muon
import scvi
import torch

/home/dags/miniconda3/envs/totalvi_env/lib/python3.10/site-packages/muon/_core/preproc.py:31: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  if Version(scanpy.__version__) < Version("1.10"):


## 2. Check package versions

In [3]:
print("scanpy:", sc.__version__)
print("scvi-tools:", scvi.__version__)
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

/tmp/ipykernel_18805/1994306271.py:1: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  print("scanpy:", sc.__version__)


scanpy: 1.11.5
scvi-tools: 1.3.3
torch: 2.12.0+cu130
CUDA available: True


'/home/dags/colabs/total-citseq-analysis/notebooks'

## 3. Load 10x CITE-seq data

In [4]:
raw_data = sc.read_10x_h5(
    "../data/raw/malt_10k_protein_v3_filtered_feature_bc_matrix.h5",
    gex_only=False
)

raw_data.var_names_make_unique()

/home/dags/miniconda3/envs/totalvi_env/lib/python3.10/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


AnnData object with n_obs × n_vars = 8412 × 33555
    var: 'gene_ids', 'feature_types', 'genome', 'pattern', 'read', 'sequence'

## 4. Check feature types

In [5]:
raw_data.var["feature_types"].value_counts()

feature_types
Gene Expression     33538
Antibody Capture       17
Name: count, dtype: int64

## 5. Separate RNA and ADT protein features and create MuData object

In [12]:
raw_data.var_names_make_unique()

rna_data = raw_data[:, raw_data.var["feature_types"] == "Gene Expression"].copy()
protein_data = raw_data[:, raw_data.var["feature_types"] == "Antibody Capture"].copy()

print("RNA data:", rna_data.shape)
print("Protein:", mdata.mod["prot"].shape)

protein_data.var_names.tolist()


RNA data: (8412, 33538)
Protein: (8412, 17)


['CD3_TotalSeqB',
 'CD4_TotalSeqB',
 'CD8a_TotalSeqB',
 'CD14_TotalSeqB',
 'CD15_TotalSeqB',
 'CD16_TotalSeqB',
 'CD56_TotalSeqB',
 'CD19_TotalSeqB',
 'CD25_TotalSeqB',
 'CD45RA_TotalSeqB',
 'CD45RO_TotalSeqB',
 'PD-1_TotalSeqB',
 'TIGIT_TotalSeqB',
 'CD127_TotalSeqB',
 'IgG2a_control_TotalSeqB',
 'IgG1_control_TotalSeqB',
 'IgG2b_control_TotalSeqB']

## 6. Create MuData object

In [13]:
mdata = muon.MuData({
    "rna": rna_data,
    "prot": protein_data
})

mdata

/home/dags/miniconda3/envs/totalvi_env/lib/python3.10/site-packages/mudata/_core/mudata.py:1416: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/home/dags/miniconda3/envs/totalvi_env/lib/python3.10/site-packages/mudata/_core/mudata.py:1272: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)


MuData object with n_obs × n_vars = 8412 × 33555
  var:	'gene_ids', 'feature_types', 'genome', 'pattern', 'read', 'sequence'
  2 modalities
    rna:	8412 × 33538
      var:	'gene_ids', 'feature_types', 'genome', 'pattern', 'read', 'sequence'
    prot:	8412 × 17
      var:	'gene_ids', 'feature_types', 'genome', 'pattern', 'read', 'sequence'

## 7. Store raw RNA counts

In [14]:
mdata.mod["rna"].layers["counts"] = mdata.mod["rna"].X.copy()

mdata.mod["rna"].layers.keys()

print("RNA counts layer:", mdata.mod["rna"].layers["counts"].shape)
print("RNA matrix:", mdata.mod["rna"].X.shape)
print("Protein matrix:", mdata.mod["prot"].X.shape)

RNA counts layer: (8412, 33538)
RNA matrix: (8412, 33538)
Protein matrix: (8412, 17)


## 8. Save preprocessed MuData object

In [ ]:
mdata.write("../data/processed/malt_citeseq_mudata.h5mu")